<a href="https://colab.research.google.com/github/himanshu-gangwar/AI_Learning/blob/main/langchain_agents_groq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 LangChain **Agents** + Groq: Research Report Generator
## Same Problem — Built with Real Agents (ReAct + Tools)

---

### 🎯 What's Different Here vs the LangChain LCEL Chains Version?

| Concept | LCEL Chains (Previous Notebook) | LangChain Agents (This Notebook) |
|---------|----------------------------------|-----------------------------------|
| **Execution model** | Linear: prompt → LLM → parser | Reasoning loop: Think → Act → Observe → Repeat |
| **Tool use** | None — LLM only generates text | Agents can call tools (search, calculator, APIs) |
| **Context passing** | Manual — you inject `{research_output}` | Automatic — agent manages scratchpad internally |
| **Control flow** | Fixed pipeline, you write every step | Agent decides how many steps and which tools |
| **Framework** | LCEL pipe (`prompt | llm | parser`) | `create_react_agent` + `AgentExecutor` |
| **Mental model** | Assembly line | Autonomous reasoning employee |
| **Lines of code** | ~80 lines | ~100 lines (but more powerful) |

### 🏗️ What We're Building
The **same 3-agent Research Report Generator**:
1. 🔍 **Researcher Agent** — Uses a Wikipedia tool to gather real facts  
2. 🧠 **Analyst Agent** — Reasons over research to extract insights  
3. ✍️ **Writer Agent** — Synthesises everything into a polished Markdown report

But this time, **agents can call tools** and **decide their own reasoning steps** — not just generate text.

> **Why learn Agents vs Chains?**  
> Chains are great for deterministic pipelines. Agents are essential when tasks require dynamic tool use, conditional reasoning, or unknown numbers of steps. Most production AI systems need both.

---
## 📦 Step 1: Install Dependencies

LangChain Agents need a few extra packages beyond the LCEL chains version:
- `langchain` — core abstractions (chains, prompts, memory, **agents**)
- `langchain-groq` — the ChatGroq LLM integration
- `langchain-community` — community tools including **WikipediaQueryRun**
- `wikipedia` — the Wikipedia API Python client (used by the tool)

```
LCEL Chains needed:   langchain  langchain-groq  langchain-core
Agents also need:  +  langchain-community  wikipedia
```

In [ ]:
!pip install langchain langchain-groq langchain-core langchain-community wikipedia -q

import langchain, langchain_groq
print(f"✅ langchain:           {langchain.__version__}")
print(f"✅ langchain-groq:      {langchain_groq.__version__}")

import langchain_community
print(f"✅ langchain-community: {langchain_community.__version__}")

---
## 🔑 Step 2: Configure API Key & LLM

Same as the chains version — we directly instantiate a `ChatGroq` object.

> **Important for Agents:** The LLM must support **function calling / tool use**.
> `llama-3.3-70b-versatile` from Groq does support this. Not all models do.

In [ ]:
import os
from getpass import getpass
from langchain_groq import ChatGroq

# Securely enter API key
groq_api_key = getpass("🔑 Enter your Groq API Key: ")
os.environ["GROQ_API_KEY"] = groq_api_key

# ─────────────────────────────────────────────────────────
# 🔑 KEY DIFFERENCE FROM LCEL CHAINS VERSION:
# ─────────────────────────────────────────────────────────
# For agents, the LLM MUST support tool calling.
# We set temperature=0 for more deterministic agent reasoning.
# Chains can use higher temperature for creative outputs;
# Agents benefit from lower temperature for reliable tool selection.
# ─────────────────────────────────────────────────────────

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,          # Lower = more reliable tool-calling decisions
    groq_api_key=groq_api_key
)

# Quick test
from langchain_core.messages import HumanMessage
test = llm.invoke([HumanMessage(content="Reply with exactly: LangChain Agents + Groq is working!")])
print(f"✅ LLM test: {test.content}")

---
## 🧰 Step 3: Understanding LangChain Agent Architecture

Before building agents, understand how they differ from chains:

```
┌─────────────────────────────────────────────────────────────────┐
│                    LCEL CHAIN (Previous Notebook)               │
│                                                                 │
│   Input → PromptTemplate → LLM → StrOutputParser → Output      │
│                                                                 │
│   Fixed, linear, one pass. No tool use. Deterministic.         │
└─────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────┐
│                    LANGCHAIN AGENT (This Notebook)              │
│                                                                 │
│   Input → Agent (LLM + Prompt) ──┐                             │
│               ↑                   ↓                             │
│          Observation         Tool Action?                       │
│               │                   │                             │
│               └──── Tool Call ←───┘   (repeats until done)     │
│                                   ↓                             │
│                             Final Answer                        │
│                                                                 │
│   Dynamic, iterative, can call tools N times.                  │
└─────────────────────────────────────────────────────────────────┘
```

### The 4 Components of a LangChain Agent

| Component | Purpose | Code |
|-----------|---------|------|
| **LLM** | The brain — does the reasoning | `ChatGroq(...)` |
| **Tools** | What the agent can DO | `[WikipediaQueryRun(...)]` |
| **Prompt** | System instructions + ReAct template | `hub.pull("hwchase17/react")` |
| **AgentExecutor** | The runtime loop | `AgentExecutor(agent, tools)` |

### ReAct Pattern (Reason + Act)
LangChain agents use the **ReAct** framework:
```
Thought: I need to look up X
Action: wikipedia
Action Input: "X"
Observation: [Wikipedia returns info about X]
Thought: Now I have enough info to answer
Final Answer: [complete response]
```

---
## 🔧 Step 4: Define the Tools

### Key Difference: Agents Have Tools, Chains Don't

```
LCEL Chain Researcher:              LangChain Agent Researcher:
──────────────────────────          ──────────────────────────────────
Relies purely on LLM's              Can CALL tools to get real data:
training data knowledge.            - Wikipedia lookup
                                    - Web search
No external data fetching.          - Calculator
                                    - Custom APIs
```

We'll create two tools:
1. **`wikipedia_search`** — fetches real Wikipedia content
2. **`word_counter`** — a simple custom tool (shows how to write your own)

In [ ]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain.tools import tool

# ─────────────────────────────────────────────────────────
# TOOL 1: Wikipedia Search (pre-built community tool)
# ─────────────────────────────────────────────────────────
# WikipediaAPIWrapper fetches and truncates Wikipedia articles
# top_k_results=3 → returns top 3 matching articles
# doc_content_chars_max=2000 → limits content to 2000 chars per article

wikipedia_tool = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper(
        top_k_results=3,
        doc_content_chars_max=2000
    )
)

# Inspect the tool — agents use .name and .description to decide when to call it
print(f"🔧 Tool 1: {wikipedia_tool.name}")
print(f"   Description: {wikipedia_tool.description[:120]}...")
print()

# ─────────────────────────────────────────────────────────
# TOOL 2: Custom Tool using @tool decorator
# ─────────────────────────────────────────────────────────
# The @tool decorator turns any Python function into a LangChain tool.
# The docstring IS the description the agent reads to decide when to use it.

@tool
def word_counter(text: str) -> str:
    """Counts the number of words in a given text string.
    Use this when you need to verify the length of your output."""
    count = len(text.split())
    return f"The text contains {count} words."

print(f"🔧 Tool 2: {word_counter.name}")
print(f"   Description: {word_counter.description}")
print()

# Collect tools into a list — this is what we'll hand to the agents
researcher_tools = [wikipedia_tool, word_counter]
writer_tools = [word_counter]  # Writer doesn't need Wikipedia

print(f"✅ Tools ready!")
print(f"   Researcher tools: {[t.name for t in researcher_tools]}")
print(f"   Writer tools:     {[t.name for t in writer_tools]}")

---
## 🔍 Step 5: Building the Researcher Agent

### How a LangChain Agent is Built

```
LCEL Chain Researcher:          LangChain Agent Researcher:
──────────────────────────      ──────────────────────────────────
researcher_chain =              agent = create_react_agent(
  researcher_prompt               llm=llm,
  | llm                           tools=[wikipedia_tool],
  | StrOutputParser()             prompt=react_prompt
                                )
# Fixed pipeline               executor = AgentExecutor(
# One LLM call                   agent=agent,
# No tools                       tools=[wikipedia_tool]
                                )
                                # Dynamic reasoning loop
                                # Multiple LLM calls possible
                                # Can call Wikipedia tool
```

### The ReAct Prompt
We use the standard `hwchase17/react` prompt from LangChain Hub — it includes the Thought/Action/Observation template that guides the model to reason step by step.

In [ ]:
from langchain import hub
from langchain.agents import create_react_agent, AgentExecutor
from langchain_core.prompts import PromptTemplate

# ─────────────────────────────────────────────────────────
# Option A: Pull the standard ReAct prompt from LangChain Hub
# This is the community-standard ReAct template.
# ─────────────────────────────────────────────────────────
# base_react_prompt = hub.pull("hwchase17/react")

# ─────────────────────────────────────────────────────────
# Option B: Define a custom ReAct prompt (used here so we
# can see exactly what the agent receives).
# ─────────────────────────────────────────────────────────
# IMPORTANT: A ReAct prompt MUST contain these variables:
#   {tools}         — list of available tools
#   {tool_names}    — comma-separated tool names
#   {input}         — the user's task
#   {agent_scratchpad} — the Thought/Action/Observation history

researcher_react_template = """You are a Senior Research Specialist with 15 years of experience \
across technology, science, and business domains.

Your goal is to conduct thorough research on any topic using available tools to fetch \
real, accurate information. Always use the Wikipedia tool to gather facts.

Structure your final research brief with these 6 sections:
1. Core Concepts & Fundamentals
2. Current State & Recent Developments
3. Key Players, Companies & Researchers
4. Real-World Applications & Use Cases
5. Challenges & Limitations
6. Future Outlook & Predictions

You have access to the following tools:
{tools}

Use the following format EXACTLY:
Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now have enough information to write a comprehensive research brief
Final Answer: [Your complete 600-800 word research brief with all 6 sections]

Begin!

Question: {input}
Thought: {agent_scratchpad}"""

researcher_prompt = PromptTemplate.from_template(researcher_react_template)

# ─────────────────────────────────────────────────────────
# Build the Researcher Agent
# create_react_agent combines LLM + tools + prompt into an agent
# AgentExecutor is the runtime that runs the Thought/Act/Observe loop
# ─────────────────────────────────────────────────────────

researcher_agent = create_react_agent(
    llm=llm,
    tools=researcher_tools,
    prompt=researcher_prompt
)

researcher_executor = AgentExecutor(
    agent=researcher_agent,
    tools=researcher_tools,
    verbose=True,           # Shows the Thought/Action/Observation trace
    max_iterations=6,       # Safety limit — prevents infinite loops
    handle_parsing_errors=True  # Gracefully handles LLM output format errors
)

print("✅ Researcher Agent built!")
print(f"   Tools available: {[t.name for t in researcher_tools]}")
print(f"   Max iterations:  {researcher_executor.max_iterations}")
print(f"   Verbose mode:    {researcher_executor.verbose} (you'll see the ReAct trace)")

---
## 🧠 Step 6: Building the Analyst Agent

### Context Passing: Agent vs Chain

In the **LCEL chains** version:
```python
# You manually inject Stage 1's output:
analyst_chain.invoke({
    "topic": topic,
    "research_output": research_output   # ← explicit injection
})
```

In this **agents** version:
```python
# You still pass context manually between agents,
# but WITHIN each agent's reasoning, it manages its own scratchpad.
analyst_executor.invoke({
    "input": f"Analyse this research on {topic}:\n{research_output}"
})
```

> **Note:** Inter-agent context passing is still manual in both approaches.
> True automatic cross-agent memory requires **LangGraph** or **memory modules** — covered in Step 10.

In [ ]:
# ─────────────────────────────────────────────────────────
# 🧠 AGENT 2: Analyst
# ─────────────────────────────────────────────────────────
# The Analyst doesn't need Wikipedia — it reasons over
# the research output already gathered by the Researcher.
# It only has the word_counter tool.
# ─────────────────────────────────────────────────────────

analyst_tools = [word_counter]

analyst_react_template = """You are a Strategic Analyst with a background in consulting and data science.

Your goal is to analyse research findings, identify key patterns and insights, \
evaluate implications, and produce structured analysis with actionable takeaways.

Always structure your analysis with these 6 sections:
1. Key Insights (top 3-5 takeaways)
2. Opportunities
3. Risks & Challenges
4. Stakeholder Impact
5. Timeline Assessment (short-term 1-2yr vs long-term 5-10yr)
6. Recommended Actions

You have access to the following tools:
{tools}

Use the following format EXACTLY:
Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: [Your complete 400-600 word strategic analysis covering all 6 sections]

Begin!

Question: {input}
Thought: {agent_scratchpad}"""

analyst_prompt = PromptTemplate.from_template(analyst_react_template)

analyst_agent = create_react_agent(
    llm=llm,
    tools=analyst_tools,
    prompt=analyst_prompt
)

analyst_executor = AgentExecutor(
    agent=analyst_agent,
    tools=analyst_tools,
    verbose=True,
    max_iterations=4,
    handle_parsing_errors=True
)

print("✅ Analyst Agent built!")
print(f"   Tools available: {[t.name for t in analyst_tools]}")
print("   Note: No Wikipedia needed — analyst reasons over provided research")

---
## ✍️ Step 7: Building the Writer Agent

The Writer synthesises both the research and analysis into a final polished report.
It uses `word_counter` to verify its output length meets the target.

In [ ]:
# ─────────────────────────────────────────────────────────
# ✍️ AGENT 3: Writer
# ─────────────────────────────────────────────────────────
# Takes both research + analysis as context.
# Uses word_counter to verify the report meets the target length.
# ─────────────────────────────────────────────────────────

writer_react_template = """You are an experienced Content Writer & Editor who has written hundreds of \
reports, whitepapers, and articles.

Your goal is to transform research and analysis into compelling, well-structured Markdown reports \
appropriate for a professional non-specialist audience.

Your writing style is clear, engaging, and authoritative. You always:
- Begin with a compelling executive summary
- Use clear section headings (## Heading)
- Explain technical concepts with concrete examples
- End with a forward-looking conclusion
- Format in Markdown (##, **, bullet points)

You have access to the following tools:
{tools}

Use the following format EXACTLY:
Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: [Your complete polished Markdown report, 800-1000 words]

Begin!

Question: {input}
Thought: {agent_scratchpad}"""

writer_prompt = PromptTemplate.from_template(writer_react_template)

writer_agent = create_react_agent(
    llm=llm,
    tools=writer_tools,
    prompt=writer_prompt
)

writer_executor = AgentExecutor(
    agent=writer_agent,
    tools=writer_tools,
    verbose=True,
    max_iterations=4,
    handle_parsing_errors=True
)

print("✅ Writer Agent built!")
print(f"   Tools available: {[t.name for t in writer_tools]}")
print("\n🎉 All 3 Agent Executors ready!")

---
## 🔗 Step 8: Building the Pipeline (Orchestrating 3 Agents)

### The Pipeline Comparison

```
LCEL Chains Pipeline:                    Agents Pipeline:
─────────────────────────────────        ─────────────────────────────────────
research = researcher_chain              research = researcher_executor.invoke(
             .invoke({"topic": topic})     {"input": f"Research {topic}"}
                                         )["output"]   ← .invoke() returns dict
analysis = analyst_chain                
             .invoke({                   analysis = analyst_executor.invoke(
               "topic": topic,             {"input": f"Analyse:\n{research}"}
               "research_output": research  )["output"]
             })

# Chains return strings directly    # Agents return {"output": str, ...}
# research IS the string            # Must extract ["output"] key
```

> **Key API difference:** `chain.invoke()` returns a string. `executor.invoke()` returns a dict with an `"output"` key.

In [ ]:
import time

def run_agent_pipeline(topic: str, verbose_pipeline: bool = True) -> dict:
    """
    Runs the full 3-agent pipeline using AgentExecutors:
    Researcher → Analyst → Writer

    Key difference from the LCEL chains pipeline:
    - Each .invoke() returns a dict, not a string
    - Must extract result["output"] at each stage
    - Each agent may make multiple LLM + tool calls internally
    """
    results = {}
    start = time.time()

    # ─── STAGE 1: Researcher Agent ───────────────────────────
    if verbose_pipeline:
        print("=" * 70)
        print("🔍 STAGE 1/3: Researcher Agent")
        print("   (watch the ReAct trace below — Thought/Action/Observation)")
        print("=" * 70)

    # ⚠️ DIFFERENCE FROM CHAINS: input key is "input", result is dict
    research_result = researcher_executor.invoke({
        "input": f"Conduct comprehensive research on this topic: {topic}. "
                 f"Use Wikipedia to gather real facts. "
                 f"Compile findings into a detailed research brief (600-800 words) "
                 f"covering all 6 required sections."
    })
    research_output = research_result["output"]  # Extract string from dict
    results["research"] = research_output

    if verbose_pipeline:
        print(f"\n✅ Research complete ({len(research_output.split())} words)")
        print()

    # ─── STAGE 2: Analyst Agent ──────────────────────────────
    if verbose_pipeline:
        print("=" * 70)
        print("🧠 STAGE 2/3: Analyst Agent")
        print("=" * 70)

    analysis_result = analyst_executor.invoke({
        "input": f"Analyse the following research on the topic: '{topic}'\n\n"
                 f"--- RESEARCH BRIEF ---\n"
                 f"{research_output}\n"
                 f"--- END RESEARCH BRIEF ---\n\n"
                 f"Produce a structured strategic analysis (400-600 words) covering all 6 sections."
    })
    analysis_output = analysis_result["output"]
    results["analysis"] = analysis_output

    if verbose_pipeline:
        print(f"\n✅ Analysis complete ({len(analysis_output.split())} words)")
        print()

    # ─── STAGE 3: Writer Agent ───────────────────────────────
    if verbose_pipeline:
        print("=" * 70)
        print("✍️  STAGE 3/3: Writer Agent")
        print("=" * 70)

    report_result = writer_executor.invoke({
        "input": f"Write a comprehensive Markdown report on '{topic}' "
                 f"using the research and analysis provided below.\n\n"
                 f"--- RESEARCH ---\n{research_output}\n\n"
                 f"--- ANALYSIS ---\n{analysis_output}\n"
                 f"--- END ---\n\n"
                 f"Use word_counter to verify your report is 800-1000 words. "
                 f"Produce a polished Markdown report with title, executive summary, "
                 f"4-5 body sections, and a forward-looking conclusion."
    })
    report_output = report_result["output"]
    results["report"] = report_output

    duration = round(time.time() - start, 1)
    results["duration"] = duration

    if verbose_pipeline:
        print(f"\n✅ Report complete ({len(report_output.split())} words)")
        print()
        print("=" * 70)
        print(f"🏁 Agent pipeline complete in {duration}s")
        print("=" * 70)

    return results


print("✅ Agent pipeline function defined!")
print("   run_agent_pipeline(topic) will run all 3 agent executors")

---
## 🚀 Step 9: Run the Agent Pipeline!

> ⏱️ Takes 60–120 seconds — agents make multiple LLM + tool calls internally.  
> Watch the `verbose=True` output to see the **ReAct trace** (Thought / Action / Observation).

In [ ]:
# ─── Set your topic ────────────────────────────────────────
TOPIC = "LinkedIn"
# ───────────────────────────────────────────────────────────

print(f"📌 Topic: {TOPIC}")
print("=" * 70)
print()

# Run the full agent pipeline
results = run_agent_pipeline(TOPIC, verbose_pipeline=True)

In [ ]:
# Inspect Stage 1 output
print("\n" + "=" * 70)
print("📋 STAGE 1 OUTPUT — Raw Research Brief:")
print("=" * 70)
print(results["research"])

In [ ]:
print("=" * 70)
print("🧠 STAGE 2 OUTPUT — Strategic Analysis:")
print("=" * 70)
print(results["analysis"])

In [ ]:
# Render the final report as formatted Markdown
from IPython.display import Markdown, display

print("✍️  FINAL REPORT (Rendered Markdown):")
display(Markdown(results["report"]))

In [ ]:
# Save all outputs to files
safe_topic = TOPIC.replace(" ", "_").replace("/", "-")

with open(f"{safe_topic}_agent_research.md", "w") as f:
    f.write(f"# Research Brief: {TOPIC}\n\n" + results["research"])

with open(f"{safe_topic}_agent_analysis.md", "w") as f:
    f.write(f"# Analysis: {TOPIC}\n\n" + results["analysis"])

with open(f"{safe_topic}_agent_report.md", "w") as f:
    f.write(results["report"])

print(f"✅ Saved 3 files:")
print(f"   {safe_topic}_agent_research.md")
print(f"   {safe_topic}_agent_analysis.md")
print(f"   {safe_topic}_agent_report.md")

---
## ⚡ Step 10: BONUS — Adding Memory to Agents

Unlike the LCEL chains version, agents can be enhanced with **conversation memory**  
so they remember previous interactions within a session.

```
LCEL Chain Memory:           LangChain Agent Memory:
──────────────────────       ──────────────────────────────────────
ConversationBufferMemory     ConversationBufferMemory works the
can be added to chains       same way, but is plugged into the
as a RunnableWithHistory.    AgentExecutor instead.
```

In [ ]:
from langchain.memory import ConversationBufferMemory

# ─────────────────────────────────────────────────────────
# BONUS: Agent with Memory
# ─────────────────────────────────────────────────────────
# ConversationBufferMemory stores the full conversation history.
# memory_key="chat_history" — the key injected into the prompt.
# return_messages=True — returns as message objects (needed for chat models).

memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

# Plug memory into a new AgentExecutor
# Note: the prompt must include a {chat_history} variable for memory to work
researcher_with_memory = AgentExecutor(
    agent=researcher_agent,
    tools=researcher_tools,
    memory=memory,
    verbose=False,
    max_iterations=6,
    handle_parsing_errors=True
)

print("✅ Agent with memory created!")
print("   Memory type: ConversationBufferMemory")
print("   The agent will remember all previous exchanges in this session.")
print()
print("💡 TIP: For persistent memory across sessions, use:")
print("   - RedisChatMessageHistory (Redis backend)")
print("   - FileChatMessageHistory (local file)")
print("   - DynamoDBChatMessageHistory (AWS)")

---
## 📊 Step 11: Full Three-Way Comparison

Now that you've seen all three approaches, here's the complete picture:

In [ ]:
comparison = """
╔══════════════════════╦═════════════════════════╦══════════════════════════╦══════════════════════════╗
║ Feature              ║ CrewAI                  ║ LangChain LCEL Chains    ║ LangChain Agents         ║
╠══════════════════════╬═════════════════════════╬══════════════════════════╬══════════════════════════╣
║ Agent definition     ║ Agent(role=,goal=,      ║ ChatPromptTemplate with  ║ create_react_agent(      ║
║                      ║   backstory=)           ║ system message           ║   llm, tools, prompt)    ║
╠══════════════════════╬═════════════════════════╬══════════════════════════╬══════════════════════════╣
║ Execution model      ║ Sequential tasks        ║ Fixed linear pipeline    ║ ReAct reasoning loop     ║
║                      ║ hidden from you         ║ you write every step     ║ agent decides steps      ║
╠══════════════════════╬═════════════════════════╬══════════════════════════╬══════════════════════════╣
║ Tool use             ║ crewai-tools prebuilt   ║ None (LLM only)          ║ @tool or community tools ║
║                      ║ or custom tools         ║                          ║ Full tool calling        ║
╠══════════════════════╬═════════════════════════╬══════════════════════════╬══════════════════════════╣
║ Context passing      ║ context=[task1]         ║ Manual {variable}        ║ Manual input string      ║
║                      ║ auto injection          ║ injection in prompt      ║ + internal scratchpad    ║
╠══════════════════════╬═════════════════════════╬══════════════════════════╬══════════════════════════╣
║ LLM calls per agent  ║ 1 (usually)             ║ Always exactly 1         ║ 1 to N (dynamic)         ║
╠══════════════════════╬═════════════════════════╬══════════════════════════╬══════════════════════════╣
║ Streaming            ║ Needs extra config      ║ .stream() built-in       ║ .stream() supported      ║
╠══════════════════════╬═════════════════════════╬══════════════════════════╬══════════════════════════╣
║ Memory               ║ memory=True w/ config   ║ RunnableWithHistory      ║ AgentExecutor(memory=)   ║
╠══════════════════════╬═════════════════════════╬══════════════════════════╬══════════════════════════╣
║ Observability        ║ Built-in task logs      ║ LangSmith tracing        ║ verbose=True ReAct trace ║
╠══════════════════════╬═════════════════════════╬══════════════════════════╬══════════════════════════╣
║ Lines of code        ║ ~30 lines               ║ ~80 lines                ║ ~100 lines               ║
╠══════════════════════╬═════════════════════════╬══════════════════════════╬══════════════════════════╣
║ Data source          ║ LLM knowledge only      ║ LLM knowledge only       ║ LLM + real tool data     ║
╠══════════════════════╬═════════════════════════╬══════════════════════════╬══════════════════════════╣
║ Best for             ║ Quick prototypes,       ║ Deterministic workflows, ║ Dynamic tasks needing    ║
║                      ║ role-based pipelines    ║ teaching LLM concepts    ║ real tool use or         ║
║                      ║                         ║ production pipelines     ║ conditional logic        ║
╚══════════════════════╩═════════════════════════╩══════════════════════════╩══════════════════════════╝
"""
print(comparison)

---
## 🧩 Step 12: Core Concepts Summary

| LangChain Agent Primitive | What it does | Code |
|---------------------------|-------------|------|
| `@tool` decorator | Turns a Python function into an agent tool | `@tool\ndef my_func(input: str) -> str:` |
| `WikipediaQueryRun` | Pre-built tool for Wikipedia searches | `WikipediaQueryRun(api_wrapper=...)` |
| `create_react_agent` | Creates an agent using ReAct reasoning pattern | `create_react_agent(llm, tools, prompt)` |
| `AgentExecutor` | The runtime loop — runs Thought/Act/Observe | `AgentExecutor(agent, tools, verbose=True)` |
| `max_iterations` | Safety limit to prevent infinite loops | `AgentExecutor(max_iterations=6)` |
| `handle_parsing_errors` | Graceful recovery when LLM format breaks | `AgentExecutor(handle_parsing_errors=True)` |
| `result["output"]` | How you extract the agent's final answer | `result = executor.invoke({...})["output"]` |

### 🔑 The Three Golden Rules for LangChain Agents

> **Rule 1:** Your ReAct prompt MUST include `{tools}`, `{tool_names}`, `{input}`, and `{agent_scratchpad}`.  
> **Rule 2:** `executor.invoke()` returns a **dict** — always extract `result["output"]`.  
> **Rule 3:** Always set `max_iterations` and `handle_parsing_errors=True` in production.

### 🚀 What to Build Next
- **LangGraph** — for non-linear agent workflows (branching, parallel agents, loops)
- **Tool calling agents** — use `create_tool_calling_agent` instead of ReAct for cleaner tool use with modern models
- **Structured output agents** — use `with_structured_output()` for JSON responses
- **LangSmith** — for full observability and tracing of every agent step
- **Multi-agent orchestration** — connect agents via LangGraph's `StateGraph`

---
> **Resources:**  
> - [LangChain Agents Docs](https://python.langchain.com/docs/modules/agents)  
> - [ReAct Agent Guide](https://python.langchain.com/docs/modules/agents/agent_types/react)  
> - [LangChain Tools](https://python.langchain.com/docs/integrations/tools)  
> - [LangGraph (advanced multi-agent)](https://langchain-ai.github.io/langgraph)  
> - [LangSmith (observability)](https://smith.langchain.com)

In [ ]:
# ─────────────────────────────────────────────────────────
# 🎓 QUICK QUIZ — Test your understanding!
# ─────────────────────────────────────────────────────────

quiz = [
    ("Q1", "What is the key architectural difference between an LCEL chain and a LangChain agent?",
     "Chains are fixed linear pipelines (1 LLM call). Agents use a ReAct loop and can make multiple LLM + tool calls dynamically."),

    ("Q2", "What 4 variables MUST a ReAct prompt template include?",
     "{tools}, {tool_names}, {input}, and {agent_scratchpad}"),

    ("Q3", "How do you extract the final answer from an AgentExecutor.invoke() result?",
     "result['output'] — invoke() returns a dict, not a string like chains do"),

    ("Q4", "What does the @tool decorator do?",
     "Converts any Python function into a LangChain tool. The docstring becomes the agent's description of when to use it."),

    ("Q5", "Why should you always set max_iterations on an AgentExecutor?",
     "Without it, a confused agent can loop forever. max_iterations is a safety ceiling on reasoning steps."),

    ("Q6", "When should you use an Agent vs a Chain?",
     "Agent: when the task needs tools, external data, or dynamic reasoning. Chain: when the pipeline is fixed and deterministic."),
]

print("🎓 QUICK QUIZ — Cover the answers and test yourself!")
print("=" * 70)
for q_id, question, answer in quiz:
    print(f"\n{q_id}: {question}")
    print(f"   💡 {answer}")